# DiT runs on rooms of furniture items (Colab)

Prepare the VM, train one run, choose the guidance weight, generate test split 0 and score it.
One cell is one blocking command; every cell can be run again (finished steps are skipped).
See `notebooks/README.md` for the four experiments of the results table and the settings that reproduce them.

In [ ]:
# 0) Settings: the only cell to edit
RUN = "dit_embed_f"     # dit_embed_none / dit_embed_f / dit_members_f (also dit_embed_l / dit_members_l)
CFG = 5.0               # None = choose again on the 100 validation rooms; the recorded weights: none 2.5, embed_f 5.0, members_f 5.0
CODE_COMMIT = ""        # full 40-character hash of the code commit (pinned in a separate commit, see README)
HF_REPO = "Dalby123/heterogeneous-diffusion"
CFG_ARG = "" if CFG is None else f"--cfg {CFG}"
assert len(CODE_COMMIT) == 40, "set CODE_COMMIT to the full 40-character hash"

In [ ]:
# 1) Hugging Face token from the Colab secret HF_TOKEN (written to the token file that the scripts read)
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
# 2) Code at the pinned commit
!rm -rf /content/GroupDiff && git clone -q https://github.com/Dalby1379/GroupDiff /content/GroupDiff && git -C /content/GroupDiff checkout -q {CODE_COMMIT} && test "$(git -C /content/GroupDiff rev-parse HEAD)" = "{CODE_COMMIT}" && git -C /content/GroupDiff log --oneline -1 || echo "[ERR] the code is not at CODE_COMMIT: do not run the next cells"

In [ ]:
# 3) Preparation: dependencies, CPU checks, room files and test gallery, feature cache, DiT-XL/2 weights (all checked by sha256)
!python /content/GroupDiff/notebooks/setup_vm.py --hf_repo {HF_REPO}

## Training

The smoke run takes 20 updates and removes its branch and outputs. The run uploads checkpoints to the branch `run-<RUN>`;
if the connection is lost, run the cell again and it continues from the latest checkpoint. A branch that already holds
`final_ema.pt` is not trained again.

In [ ]:
# 4) Smoke run
!cd /content/GroupDiff && python notebooks/train_run.py --run {RUN} --smoke --hf_repo {HF_REPO}

In [ ]:
# 5) Training (about 12 hours on an A100 40GB)
!cd /content/GroupDiff && python notebooks/train_run.py --run {RUN} --hf_repo {HF_REPO}

## Evaluation

In [ ]:
# 6) Guidance weight (or the fixed CFG), test split 0, clip embeddings, ranks against the test gallery
!cd /content/GroupDiff && python notebooks/eval_run.py --run {RUN} {CFG_ARG} --hf_repo {HF_REPO}

In [ ]:
# 7) Results of test split 0
import glob, json
tag = "" if CFG is None else f"_cfg{float(CFG)}"
for path in sorted(glob.glob(f"/content/eval/{RUN}{tag}/test0_members*/summary.json")):
    s = json.load(open(path))
    print(path.split("/")[-2], "  ".join(f"{k} {s[k]:.2f}" for k in ("recall_clip@1", "recall_clip@5", "recall_clip@10", "recall_clip@20", "category_correct")))

In [ ]:
# 8) Optional: put the results on the branch eval-<RUN> (never an existing branch)
!cd /content/GroupDiff && python notebooks/eval_run.py --run {RUN} {CFG_ARG} --hf_repo {HF_REPO} --upload